# 🔢 NumPy Special — Power User Techniques

Beyond the basics — advanced NumPy patterns used in ML, signal processing, and scientific computing.

**Topics:**
1. Broadcasting rules
2. Advanced indexing — fancy, boolean, take
3. Stride tricks and views
4. Universal functions (ufuncs)
5. Structured arrays
6. Memory layout — C vs Fortran order
7. Vectorised operations vs loops (benchmarks)
8. FFT — signal processing
9. Random number generation (new Generator API)
10. NumPy in ML — batch matrix ops, einsum

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import time
from numpy.fft import fft, fftfreq

# New random Generator API (numpy >= 1.17) — more flexible than np.random.*
rng = np.random.default_rng(seed=42)

np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(f'NumPy version: {np.__version__}')

## 1. Broadcasting — The Core NumPy Superpower

Broadcasting lets you operate on arrays of different shapes without copying data.
**Rule:** starting from the trailing dimensions, sizes must be equal or one of them is 1.

In [ ]:
# ── Broadcasting ─────────────────────────────────────────────────────────────

# Example 1: add a row vector to a column vector → outer sum (no loops!)
rows = np.array([[0], [10], [20], [30]])   # shape (4, 1)
cols = np.array([1, 2, 3, 4, 5])          # shape (5,) → treated as (1, 5)
# Broadcasting: (4,1) + (5,) → (4,5)
table = rows + cols
print('Addition table (4×5):'); print(table)

# Example 2: standardise a 2-D data matrix (subtract mean and divide by std per column)
X = rng.normal(loc=[10, 100, 0.5], scale=[2, 20, 0.1], size=(100, 3))  # (100, 3)
# mean and std have shape (3,) — broadcast over 100 rows automatically
X_std = (X - X.mean(axis=0)) / X.std(axis=0)  # (100,3) - (3,) → (100,3)
print(f'\nStandardised mean:  {X_std.mean(axis=0).round(10)}')
print(f'Standardised std:   {X_std.std(axis=0).round(6)}')

# Example 3: pairwise Euclidean distance matrix (n×n) without any loops
# Using broadcasting: ||a - b||² = ||a||² + ||b||² - 2 a·b
points = rng.random((5, 2))   # 5 points in 2-D
# diff[i,j] = points[i] - points[j], shape (5, 5, 2)
diff = points[:, None, :] - points[None, :, :]
dist_matrix = np.sqrt((diff**2).sum(axis=-1))
print('\nPairwise distance matrix:')
print(dist_matrix.round(4))

In [ ]:
# ── 2. Advanced Indexing ──────────────────────────────────────────────────────

A = np.arange(25).reshape(5, 5)  # 5×5 matrix

# Fancy indexing: select arbitrary rows and columns
rows_idx = [0, 2, 4]
cols_idx = [1, 3]
sub = A[np.ix_(rows_idx, cols_idx)]   # np.ix_ creates open mesh for Cartesian selection
print('Selected submatrix:'); print(sub)

# Boolean mask indexing
mask = (A % 3 == 0) & (A > 5)   # elements divisible by 3 AND greater than 5
print(f'Elements divisible by 3 AND > 5: {A[mask]}')

# np.where: conditional selection (vectorised if-else)
B = np.where(A % 2 == 0, A, -A)   # keep evens, negate odds
print('np.where (evens positive, odds negative):'); print(B)

# np.take: gather elements along an axis (useful for embedding lookups)
vocab = np.array(['<PAD>', 'hello', 'world', 'foo', 'bar'])
token_ids = np.array([1, 2, 1, 4])   # sequence of token ids
decoded = np.take(vocab, token_ids)   # equivalent to vocab[token_ids]
print(f'Decoded tokens: {decoded}')

# Scatter: update values at specific indices
arr = np.zeros(8, dtype=int)
np.add.at(arr, [0, 2, 2, 5], 1)   # increment indices (supports duplicate indices)
print(f'After scatter add at [0,2,2,5]: {arr}')

In [ ]:
# ── 3. Stride Tricks — Zero-Copy Rolling Windows ─────────────────────────────
from numpy.lib.stride_tricks import sliding_window_view

# Create a time series
ts = np.arange(10, dtype=float)  # [0, 1, 2, ..., 9]

# sliding_window_view: returns a view (no data copy!) of shape (n-w+1, w)
# Useful for computing rolling statistics efficiently
windows = sliding_window_view(ts, window_shape=3)  # each row is 3 consecutive elements
print('Rolling windows (size 3):'); print(windows)
print(f'Rolling mean: {windows.mean(axis=1)}')
print(f'Rolling std:  {windows.std(axis=1).round(4)}')

# More complex: 2-D image patches
image = rng.integers(0, 255, (8, 8))   # 8×8 grayscale image
patches = sliding_window_view(image, (3, 3))   # 3×3 patch views
print(f'\n2-D patches shape: {patches.shape}')  # (6, 6, 3, 3)
# Max-pool: take max of each 3×3 patch
max_pooled = patches.max(axis=(-2, -1))
print(f'Max-pooled shape: {max_pooled.shape}')  # (6, 6)

In [ ]:
# ── 4. einsum — Einstein Summation ───────────────────────────────────────────
# einsum provides a single, clear notation for any tensor contraction.
# It's also highly optimised — often faster than explicit loops.

A_mat = rng.random((3, 4))   # (3, 4)
B_mat = rng.random((4, 5))   # (4, 5)

# Matrix multiplication: 'ij,jk->ik'  (sum over j)
C_ein = np.einsum('ij,jk->ik', A_mat, B_mat)
C_dot = A_mat @ B_mat
print(f'einsum == @ : {np.allclose(C_ein, C_dot)}')

# Batch matrix multiply: 'bij,bjk->bik'
B_data = rng.random((8, 4, 4))   # batch of 8 matrices
C_data = rng.random((8, 4, 5))
result = np.einsum('bij,bjk->bik', B_data, C_data)  # (8, 4, 5)
print(f'Batch matmul shape: {result.shape}')

# Attention: Q @ K.T for a batch of heads
# (batch, heads, seq, d_k) × (batch, heads, d_k, seq) → (batch, heads, seq, seq)
B, H, T, dk = 2, 4, 8, 16
Q = rng.random((B, H, T, dk))
K = rng.random((B, H, T, dk))
scores = np.einsum('bhqd,bhkd->bhqk', Q, K) / np.sqrt(dk)  # (B, H, T, T)
print(f'Attention scores shape: {scores.shape}')   # (2, 4, 8, 8)

In [ ]:
# ── 5. FFT — Frequency Analysis ──────────────────────────────────────────────

# Simulate a signal composed of two frequencies + noise
# Real-world: detect dominant frequencies in a vibration sensor
sample_rate = 1000    # Hz
duration    = 1.0     # seconds
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# Signal: 50 Hz + 120 Hz components + random noise
signal = (2.5 * np.sin(2 * np.pi * 50 * t) +
          1.0 * np.sin(2 * np.pi * 120 * t) +
          0.5 * rng.standard_normal(len(t)))

# Compute FFT
spectrum = fft(signal)            # complex frequency spectrum
freqs    = fftfreq(len(t), 1/sample_rate)  # corresponding frequencies (Hz)

# Magnitude spectrum: |FFT| (only positive frequencies)
n_half    = len(t) // 2
magnitude = (2 / len(t)) * np.abs(spectrum[:n_half])
pos_freqs = freqs[:n_half]

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(t[:200], signal[:200], color='#60a5fa', lw=1)
axes[0].set_title('Time-domain signal (first 0.2s)')
axes[0].set_xlabel('Time (s)'); axes[0].set_ylabel('Amplitude')

axes[1].plot(pos_freqs, magnitude, color='#34d399', lw=1.5)
axes[1].set_title('Frequency spectrum')
axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Magnitude')
axes[1].set_xlim(0, 200)
# Annotate the peaks
for f, amp in [(50, 2.5), (120, 1.0)]:
    axes[1].annotate(f'{f} Hz', xy=(f, amp*0.9),
                     xytext=(f+10, amp), arrowprops=dict(arrowstyle='->', color='white'),
                     color='white', fontsize=9)
plt.tight_layout(); plt.show()

# Find the top 3 dominant frequencies
top3_idx = np.argpartition(magnitude, -3)[-3:][::-1]
print(f'Top-3 frequencies: {pos_freqs[np.argsort(magnitude)[-3:][::-1]].astype(int)} Hz')

In [ ]:
# ── 6. Performance: Vectorised vs Loop Benchmarks ────────────────────────────

N = 1_000_000
data = rng.standard_normal(N)

def python_loop_sum(arr):
    """Pure Python loop — slowest."""
    total = 0.0
    for x in arr:
        total += x
    return total

def python_listcomp(arr):
    """List comprehension with sum() — slightly faster than loop."""
    return sum(x for x in arr)

def numpy_sum(arr):
    """NumPy C-level sum — fastest."""
    return arr.sum()

# Benchmark each
benchmarks = {}
for name, fn, arg in [
    ('Python loop',   python_loop_sum,  data.tolist()),
    ('Python sum()',  python_listcomp,  data.tolist()),
    ('NumPy .sum()',  numpy_sum,        data),
]:
    t0 = time.perf_counter()
    result = fn(arg)
    elapsed = time.perf_counter() - t0
    benchmarks[name] = elapsed * 1000
    print(f'{name:20s} → {elapsed*1000:7.2f}ms  result={result:.4f}')

# Visualise speedup
fig, ax = plt.subplots(figsize=(7, 2.5))
bars = ax.barh(list(benchmarks.keys()), list(benchmarks.values()),
               color=['#ef4444','#f59e0b','#34d399'])
ax.set_xlabel('Time (ms)'); ax.set_title(f'Sum of {N:,} floats — timing comparison')
for bar, val in zip(bars, benchmarks.values()):
    ax.text(val + 1, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}ms', va='center', fontsize=9)
plt.tight_layout(); plt.show()